In [5]:
import os
import pandas as pd
from dotenv import load_dotenv
from bertopic import BERTopic

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
load_dotenv()  # Load OPENROUTER_API_KEY from .env, if present

# Choose between 'API' (OpenRouter via LiteLLM), 
# 'CPU' (llama.cpp GGUF local), 
# 'GPU' (Transformers local)
ACTIVE_BACKEND = "CPU"  # <-- Change this to test different approaches

# Optional: Set number of docs to test quickly (None for all)
TEST_DOC_LIMIT = 500


In [6]:
# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------
PARQUET_PATH = r"C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260310_180458_ADS_Curation_Run\data\publications_disambiguated.parquet"

df = pd.read_parquet(PARQUET_PATH)

if TEST_DOC_LIMIT:
    docs = df["full_text"].dropna().tolist()[:TEST_DOC_LIMIT]
else:
    docs = df["full_text"].dropna().tolist()

print(f"Loaded {len(docs)} documents.")


Loaded 382 documents.


In [7]:
# ---------------------------------------------------------
# 2. SETUP REPRESENTATION MODEL
# ---------------------------------------------------------
PHYSICS_PROMPT = """\
You are an experienced researcher in gravitational physics, astrophysics, \
and cosmology. You are labeling research topic clusters based on scientific \
abstracts.

Documents: [DOCUMENTS]
Keywords: [KEYWORDS]

Task:
- Generate EXACTLY ONE topic label.
- The label MUST contain between 4 and 7 words (inclusive).
- The label should read like a review article or textbook chapter title.

Guidelines:
- Use standard scientific terminology from the field.
- Be specific about the physical phenomenon or method.
- Avoid generic terms like "studies", "analysis", or "research".

Output format (single line):
topic: <label>

Do NOT write anything else (no explanations, no additional sentences, \
no quotes, no bullet points)."""

representation_model = None

if ACTIVE_BACKEND == "API":
    from bertopic.representation import LiteLLM
    
    # LiteLLM routes correctly when using 'openrouter/' prefix.
    # ACHTUNG: Hier MUSS ein Text-Generierungs-Modell stehen, KEIN Embedding-Modell!
    model_name = "openrouter/google/gemini-3.1-flash-lite-preview"
    print(f"Initializing API Backend (LiteLLM) with {model_name}...")
    
    representation_model = LiteLLM(model=model_name, prompt=PHYSICS_PROMPT)

elif ACTIVE_BACKEND == "CPU":
    from llama_cpp import Llama
    from bertopic.representation import LlamaCPP
    from huggingface_hub import hf_hub_download
    
    model_path = hf_hub_download(
        repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
        filename="qwen2.5-0.5b-instruct-q4_k_m.gguf",
    )
    print(f"Initializing CPU Backend (llama.cpp) with qwen2.5-0.5b-instruct-q4_k_m.gguf ...")
    
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=0,  # CPU only
        n_ctx=16384,     # Expand context window
        stop=["Q:", "\n\n"],
        verbose=False
    )
    representation_model = LlamaCPP(llm, prompt=PHYSICS_PROMPT)

elif ACTIVE_BACKEND == "GPU":
    from transformers import pipeline
    from bertopic.representation import TextGeneration
    
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"
    print(f"Initializing GPU Backend (Transformers) with {model_name}...")
    
    generator = pipeline(
        "text-generation", 
        model=model_name, 
        device_map="auto",
        max_new_tokens=50,
    )
    representation_model = TextGeneration(generator, prompt=PHYSICS_PROMPT)

else:
    raise ValueError("Invalid ACTIVE_BACKEND selected.")

Initializing CPU Backend (llama.cpp) with qwen2.5-0.5b-instruct-q4_k_m.gguf ...


llama_context: n_ctx_per_seq (16384) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [8]:
# ---------------------------------------------------------
# 3. RUN BERTOPIC
# ---------------------------------------------------------
print("Starting BERTopic training (this may take a while depending on backend)...")

topic_model = BERTopic(
    representation_model=representation_model,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

print("\n--- DONE! Top Topics ---")
print(topic_model.get_topic_info().head(10))


2026-03-11 15:34:42,995 - BERTopic - Embedding - Transforming documents to embeddings.


Starting BERTopic training (this may take a while depending on backend)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

2026-03-11 15:34:51,364 - BERTopic - Embedding - Completed ✓
2026-03-11 15:34:51,366 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 15:34:51,834 - BERTopic - Dimensionality - Completed ✓
2026-03-11 15:34:51,836 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 15:34:51,852 - BERTopic - Cluster - Completed ✓
2026-03-11 15:34:51,858 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 4/4 [01:11<00:00, 17.79s/it]
2026-03-11 15:36:03,097 - BERTopic - Representation - Completed ✓



--- DONE! Top Topics ---
   Topic  Count                                               Name  \
0     -1     32  -1_topic: Correlations in Gravitational Physic...   
1      0    310  0_topic: Mach-Einstein doctrine and general re...   
2      1     26  1_topic: Helmholtz Theorem and Its Generalizat...   
3      2     14  2_topic: The Einstein Effect in Gravitational ...   

                                      Representation  \
0  [topic: Correlations in Gravitational Physics ...   
1  [topic: Mach-Einstein doctrine and general rel...   
2  [topic: Helmholtz Theorem and Its Generalizati...   
3  [topic: The Einstein Effect in Gravitational F...   

                                 Representative_Docs  
0  [The problem of the Grand Unification Theory. ...  
1  [Aspects of the Mach-Einstein doctrine and geo...  
2  [On the General-Relativistic and Covariant For...  
3  [The relativistic enlargement of the apparent ...  
